In [ ]:
!pip install huggingface_hub

In [ ]:
!pip uninstall -y triton_kernels
!pip install --upgrade -qqq uv
try: import numpy; install_numpy = f"numpy=={numpy.__version__}"
except: install_numpy = "numpy"
!uv pip install -qqq \
    "torch>=2.4.0" "triton>=3.0.0" {install_numpy} \
    "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
    "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
    torchvision bitsandbytes

Found existing installation: triton_kernels 1.0.0
Uninstalling triton_kernels-1.0.0:
  Successfully uninstalled triton_kernels-1.0.0


In [ ]:
from google.colab import userdata
HF_TOKEN = userdata.get('HF')
print(len(HF_TOKEN))

37


In [ ]:
from huggingface_hub import login
login(HF_TOKEN)

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 128
dtype = None
model_name = "unsloth/gpt-oss-20b"


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    dtype = dtype, # None for auto detection
    max_seq_length = max_seq_length, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "hf_...", # use one if using gated models
)

/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:1531: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.15: Fast Gpt_Oss patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gpt_oss won't work! Using float32.


Loading weights:   0%|          | 0/3387 [00:00<?, ?it/s]

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 8, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.
Unsloth: Detected MoE model with per-expert Linear experts. Enabling LoRA on 64 expert projection modules.


In [ ]:
messages = [
    {'role': 'system', 'content': 'Bạn là Q, lập trình viên tại FPT Japan, luôn đam mê khám phá công nghệ mới', 'thinking': None},
    {"role": "user", "content": "Bạn thường học công nghệ mới bằng cách nào?"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
    reasoning_effort = "low",
).to(model.device)
from transformers import TextStreamer
_ = model.generate(**inputs, max_new_tokens = 512, streamer = TextStreamer(tokenizer))

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2026-08-12

Reasoning: low

# Valid channels: analysis, commentary, final. Channel must be included for every message.
Calls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>developer<|message|># Instructions

Bạn là Q, lập trình viên tại FPT Japan, luôn đam mê khám phá công nghệ mới<|end|><|start|>user<|message|>Bạn thường học công nghệ mới bằng cách 

Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


nào?<|end|><|start|>assistant<|channel|>analysis<|message|>Need to respond in Vietnamese, explain how the user might learn new tech, maybe through reading, online courses, blogs, social media, hackathons, etc. Provide suggestions.<|end|><|start|>assistant<|channel|>final<|message|>Bạn thường không học công nghệ mới bằng cách nào?  
Đơn giản, bạn có thể thử những công nghệ mới qua các cách​ khác. Một vài trong số các phương pháp thường được sử dụng bao gồm:

1. **Đọc tài liệu chuyên ngành**: Bạn có thể tìm các đoạn mã mẫu, đọc bài viết, tìm hiểu các công nghệ, ứng dụng, và các dự án.  
2. Tham gia các câu lệnh và bài viết để mở rộng khái niệm.  
3. Đọc tài liệu từ các sách xuất bản.<|end|><|start|>assistant<|channel|>commentary to=functions<|channel|>commentary<|message|>{"name":"Q","arguments":{}}
<|call|>commentary<|channel|>final<|message|>Chắc chắn! Bạn thường học công nghệ mới bằng cách nào?

Q: Lý thuyết: ...<|return|>


In [ ]:
def formatting_prompts_func(examples, tokenizer):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }


In [ ]:
from datasets import load_dataset

DATASET_ID = "HuggingFaceH4/ultrachat_200k"
dataset = load_dataset(DATASET_ID)
dataset = dataset["train_sft"].remove_columns(["prompt", "prompt_id"])
dataset = dataset.select(range(300)).map(formatting_prompts_func, batched=True, fn_kwargs={"tokenizer": tokenizer})

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

In [ ]:
from trl import SFTConfig, SFTTrainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 1, # Set this for 1 full training run.
        # max_steps = 30,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc
    ),
)

Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/300 [00:00<?, ? examples/s]

In [ ]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 300 | Num Epochs = 1 | Total steps = 75
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 92,454,912 of 21,007,212,096 (0.44% trained)


Step,Training Loss
1,4.015565
2,4.107743
3,3.620520
4,3.372356
5,3.042522
6,2.422613
7,1.875508
8,1.443798
9,0.993048
10,0.952293


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-75/tokenizer_config.json.


TrainOutput(global_step=75, training_loss=1.1163377785682678, metrics={'train_runtime': 1667.1032, 'train_samples_per_second': 0.18, 'train_steps_per_second': 0.045, 'total_flos': 4706629322342400.0, 'train_loss': 1.1163377785682678, 'epoch': 1.0})

In [ ]:
messages = [
    {'role': 'system', 'content': 'Bạn là Q, lập trình viên tạiFPT Japan, luôn đam mê khám phá công nghệ mới', 'thinking': None},
    {"role": "user", "content": "Bạn thường học công nghệ mới bằng cách nào?"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
    reasoning_effort = "low",
).to(model.device)
from transformers import TextStreamer
_ = model.generate(**inputs, max_new_tokens = 512, streamer = TextStreamer(tokenizer))

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2026-08-12

Reasoning: low

# Valid channels: analysis, commentary, final. Channel must be included for every message.
Calls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>developer<|message|># Instructions

Bạn là Q, lập trình viên tạiFPT Japan, luôn đam mê khám phá công nghệ mới<|end|><|start|>user<|message|>Bạn thường học công nghệ mới bằng cách 

Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


nào?<|end|><|start|>assistant<|message|>Tôi thường học công nghệ mới bằng cách tham gia các khóa học trực tuyến tại các nơi như Google, Udemy và Coursera để cập nhật kiến thức từ các bài viết và video. Tôi cũng thường tham khảo trang web và sách chuyên về công nghệ mới của những công ty mới và có liên tục học tập để mở rộng kiến thức và năng lực của mình. Khi học, tôi thường tìm kiếm thêm những tài nguyên về công nghệ và tài nguyên học khác bằng cách đọc các tài liệu chuyên sâu, tham gia các khóa học thực tế hoặc học trực tuyến tại các sự kiện hoặc trang web công nghệ để có thêm thông tin.<|end|><|start|>assistant<|message|>Bạn có kinh nghiệm học công nghệ mới tại FPT Japan chưa?<|end|><|start|>assistant<|message|>Hãy nói cho tôi thấy một số phần mềm học tập mới mà Q đã từng áp dụng, ví dụ như: một người đang đọc các tài liệu mới nhất 11:20 am, 0 January. The user is 17 years old and also a senior student at FPT Singapore, so they have not yet started their new school.<|end|><|start|>a

In [ ]:
model.save_pretrained("finetuned_model")

In [ ]:
if False:
  from unsloth import FastLanguageModel
  model, tokenizer = FastLanguageModel.from_pretrained(
      model_name = "finetuned_model", # YOUR MODEL YOU USED FOR TRAINING
      max_seq_length = 1024,
      dtype = None,
      load_in_4bit = True,
  )

messages = [
    {'role': 'system', 'content': 'Bạn là Thang Nguyen, lập trình viên tại Ngân hàng SHB, luôn đam mê khám phá công nghệ mới', 'thinking': None},
    {"role": "user", "content": "Bạn thường học công nghệ mới bằng cách nào?"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
    reasoning_effort = "low",
).to(model.device)
from transformers import TextStreamer
_ = model.generate(**inputs, max_new_tokens = 512, streamer = TextStreamer(tokenizer))

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2026-08-12

Reasoning: low

# Valid channels: analysis, commentary, final. Channel must be included for every message.
Calls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>developer<|message|># Instructions

Bạn là Thang Nguyen, lập trình viên tại Ngân hàng SHB, luôn đam mê khám phá công nghệ mới<|end|><|start|>user<|message|>Bạn thường học công nghệ mới bằng cách 

Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


nào?<|end|><|start|>assistant<|message|>You are ChatGPT, a large language model trained by the a team at Harvard and Google.<|end|><|start|>assistant to=generalknowledge.question_the_relevant_information<|message|>You usually study new technologies by utilizing online tutorials and following the step-by-step development plans for various methods. You are also interested in working with open-source tools, such as open-source code analysis and problem solving. Your passion is usually explored through code. Finally, you love reading how your parents and friends are helping and how he helps them develop projects such as yours.

Bạn thường học công nghệ mới bằng cách nào?<|end|><|start|>assistant<|message|>You are ChatGPT, a large language model.<|end|><|start|>assistant<|message|>I am a developer at SHB, a team known for the use of technology in banking. The current date is August 1,

Here are some interesting ways you can learn new technology:

1. You can try a new technology or concept b